In [ ]:
!pip install datasets
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [ ]:
from datasets import Dataset
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
from sklearn.metrics import accuracy_score, f1_score

# 1. Load data
df = pd.read_csv('abstract.csv').head(20)  # Using first 20 rows
dataset = Dataset.from_pandas(df)

# 2. SciBERT setup
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Correct Tokenization function (with proper label handling)
def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    # Convert labels to LONG (int64) for classification
    tokenized["label"] = [int(label) for label in examples["label"]]
    return tokenized

# 4. Prepare k-fold splits
texts = dataset["Abstract"]
labels = dataset["label"]
kf = KFold(n_splits=3, shuffle=True, random_state=42)  # Reduced folds for small data

# 5. Evaluation metric
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

for fold, (train_idx, val_idx) in enumerate(kf.split(texts)):
    print(f"\n=== Fold {fold + 1} ===")

    # Create splits with INTEGER labels
    train_data = Dataset.from_dict({
        "text": np.array(texts)[train_idx].tolist(),
        "label": np.array(labels)[train_idx].astype(np.int64)  # Critical fix: int64 instead of float
    })
    val_data = Dataset.from_dict({
        "text": np.array(texts)[val_idx].tolist(),
        "label": np.array(labels)[val_idx].astype(np.int64)    # Same fix here
    })

    # Tokenize
    tokenized_train = train_data.map(tokenize_function, batched=True)
    tokenized_val = val_data.map(tokenize_function, batched=True)

    # Initialize model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(np.unique(labels)),
        problem_type="single_label_classification"
    )

    # Training args optimized for small batches
    training_args = TrainingArguments(
        output_dir=f"./scibert_fold_{fold+1}",
        evaluation_strategy="steps",
        eval_steps=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=5,  # Effective batch size = 20
        num_train_epochs=15,
        learning_rate=2e-5,
        save_strategy="no",
        logging_steps=1,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )

    print(f"Training samples: {len(train_data)}, Validation samples: {len(val_data)}")
    trainer.train()
    results = trainer.evaluate()
    print(f"Fold {fold+1} Results:", results)


=== Fold 1 ===


Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Training samples: 13, Validation samples: 7


...

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Training samples: 13, Validation samples: 7


KeyboardInterrupt: 

In [ ]:
from datasets import Dataset
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

# 1. Enhanced Data Loading with Validation
df = pd.read_csv('abstract.csv')
head_10 = df.head(10)

# Get 10 from the tail
tail_10 = df.tail(10)

# Concatenate them
combined_df = pd.concat([head_10, tail_10]).reset_index(drop=True)
#Randomly select 20 rows
df = combined_df
print("\n=== Data Integrity Check ===")
print(f"Total samples: {len(df)}")
print("Label distribution:", Counter(df['label']))
print("Null values:\n", df.isnull().sum())

# Data Cleaning
df = df.dropna(subset=['Abstract', 'label'])
df = df[df['Abstract'].str.strip().astype(bool)]

# 2. Label Processing with Verification
print("\nOriginal label values:", sorted(df['label'].unique()))
df['label'] = pd.factorize(df['label'])[0]  # Convert to 0-indexed integers
print("Processed labels:", sorted(df['label'].unique()))

# 3. Dataset Preparation with Train/Test Split
dataset = Dataset.from_pandas(df)
texts = dataset["Abstract"]
labels = dataset["label"]

# 4. Debugging KFold Implementation
kf = KFold(n_splits=2, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(kf.split(texts, labels)):
    print(f"\n=== Fold {fold+1} === [Train: {len(train_idx)}, Val: {len(val_idx)}]")

    # Verify no overlap
    assert len(set(train_idx) & set(val_idx)) == 0, "Data leakage detected!"

    # Sample inspection
    print("Sample train labels:", [labels[i] for i in train_idx[:5]])
    print("Sample val labels:", [labels[i] for i in val_idx[:5]])

# 5. Model Setup with Debugging
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["Abstract"], padding="max_length", truncation=True, max_length=128)

# 6. Modified Training Loop with Early Stopping
for fold, (train_idx, val_idx) in enumerate(kf.split(texts, labels)):
    print(f"\n\n===== Fold {fold+1} =====")

    # Create datasets
    train_data = Dataset.from_dict({"Abstract": [texts[i] for i in train_idx],
                                     "label": [labels[i] for i in train_idx]})
    val_data = Dataset.from_dict({"Abstract": [texts[i] for i in val_idx],
                                   "label": [labels[i] for i in val_idx]})

    # Tokenize with verification
    tokenized_train = train_data.map(tokenize_function, batched=True)
    tokenized_val = val_data.map(tokenize_function, batched=True)

    print("\nFirst training sample:", tokenized_train[0])
    print("First validation sample:", tokenized_val[0])

    # Model initialization with config
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(np.unique(labels)),
        problem_type="single_label_classification"
    )

    # Conservative training parameters
    training_args = TrainingArguments(
        output_dir=f"./fold_{fold+1}",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=5,
        evaluation_strategy="steps",
        eval_steps=10,
        logging_steps=5,
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=False,
        metric_for_best_model='f1',
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    # Custom compute_metrics with debugging
    def compute_metrics(p):
        preds = p.predictions.argmax(-1)
        print("\nPrediction samples:", list(zip(p.label_ids[:5], preds[:5])))
        print("Unique Predictions:", np.unique(preds))
        return {
            'accuracy': accuracy_score(p.label_ids, preds),
            'f1': f1_score(p.label_ids, preds, average='weighted')
        }

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )

    # Training with progress monitoring
    print("\nStarting training...")
    trainer.train()

    predictions = trainer.predict(tokenized_val)
    print("Unique predictions from the validation set:", np.unique(predictions.predictions.argmax(-1)))

    # Detailed evaluation
    print("\nFinal evaluation:")
    results = trainer.evaluate()
    print({k: v for k, v in results.items() if k in ['eval_accuracy', 'eval_f1', 'eval_loss']})

    # Prediction analysis
    print("\nMisclassified examples:")
    preds = predictions.predictions.argmax(-1)
    for i in range(min(10, len(val_idx))):
        if preds[i] != labels[val_idx[i]]:
            print(f"\nText: {texts[val_idx[i]][:100]}...")
            print(f"True: {labels[val_idx[i]]}, Predicted: {preds[i]}")


=== Data Integrity Check ===
Total samples: 20
Label distribution: Counter({0: 10, 1: 10})
Null values:
 label       0
Abstract    0
dtype: int64

Original label values: [np.int64(0), np.int64(1)]
Processed labels: [np.int64(0), np.int64(1)]

=== Fold 1 === [Train: 10, Val: 10]
Sample train labels: [0, 0, 0, 0, 0]
Sample val labels: [0, 0, 0, 0, 0]

=== Fold 2 === [Train: 10, Val: 10]
Sample train labels: [0, 0, 0, 0, 0]
Sample val labels: [0, 0, 0, 0, 0]


===== Fold 1 =====


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]


First training sample: {'Abstract': 'As the complexity of our neural network models grow,  so too do the data and computation requirements for successful training.  One proposed solution to this problem is training on a distributed network of computational devices, thus distributing the computational and data storage loads.  This strategy has already seen some adoption by the likes of Google and other companies.  In this paper we propose a new method of distributed, decentralized learning that allows a network of computation nodes to coordinate their training using asynchronous updates over an unreliable network while only having access to a local dataset.  This is achieved by taking inspiration from Distributed Averaging Consensus algorithms to coordinate the various nodes.  Sharing the internal model instead of the training data allows the original raw data to remain with the computation node.  The asynchronous nature and lack of centralized coordination allows this paradigm to func

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


...


Starting training...


Step,Training Loss,Validation Loss,Accuracy,F1
10,0.323200,0.647389,0.500000,0.333333



Prediction samples: [(np.int64(0), np.int64(1)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1))]
Unique Predictions: [1]



Prediction samples: [(np.int64(0), np.int64(0)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1))]
Unique Predictions: [0 1]
Unique predictions from the validation set: [0 1]

Final evaluation:



Prediction samples: [(np.int64(0), np.int64(0)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1)), (np.int64(0), np.int64(1))]
Unique Predictions: [0 1]
{'eval_loss': 0.6150730848312378, 'eval_accuracy': 0.6, 'eval_f1': 0.5238095238095238}

Misclassified examples:

Text: Embedding logical knowledge information into text generation is a challenging NLP task. In this pape...
True: 0, Predicted: 1

Text: Post-training dropout based approaches achieve high sparsity and are well established means of decip...
True: 0, Predicted: 1

Text: Variational Auto Encoder (VAE) provide an efficient latent space representation of complex data dist...
True: 0, Predicted: 1

Text: We present a new method for searching optimal hyperparameters among several tasks and several criter...
True: 0, Predicted: 1
